# Forming Parameter Optimization Using a Neural Network Surrogate

**Goal:** find the temperature, strain rate, and strain that **minimize** predicted flow stress during hot forming.

**Method:**
1. Train a neural network to predict flow stress from (temperature, strain rate, strain), using experimental data.
2. Treat that trained network as a fast *surrogate* for the real material behaviour.
3. Search the input space (within realistic bounds) for the combination that makes the surrogate predict the lowest stress.

Upload `Data_1.xlsx` (inputs: 1/T in Kelvin, ln(strain rate), strain) and `Data_2.xlsx` (target: normalized flow stress) when prompted in Section 1.

In [ ]:
# Libraries used in this notebook:
#   pandas   - load the Excel data
#   numpy    - array math
#   sklearn  - build and train the neural network
#   scipy    - run the optimizer (SLSQP, the Python equivalent of MATLAB's SQP)
#   matplotlib - plot predicted vs. actual stress

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from scipy.optimize import minimize


## Section 1: Load the data

In [ ]:
# Upload Data_1.xlsx and Data_2.xlsx when prompted.
# (If you'd rather load from Google Drive, mount it and use pd.read_excel with a file path instead.)

from google.colab import files
uploaded = files.upload()


In [ ]:
data1 = pd.read_excel('Data 1.xlsx')   # columns: 1/T(Kelvin), ln(strain rate), strain
data2 = pd.read_excel('Data 2.xlsx')   # column: normalized flow stress (stress / max stress)

X = data1.iloc[:, :3].values   # inputs,  400 x 3
y = data2.iloc[:, -1].values   # target,  400 values

print('Inputs shape:', X.shape)
print('Target shape:', y.shape)
data1.head()


## Section 2: Build and train the neural network

In [ ]:
# Split the 400 samples: 70% train, 30% test.
# The test set is held out and never used in training -- it gives an
# honest check of accuracy afterwards.

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

# A small feedforward network with one hidden layer of 12 neurons --
# the same size used in the original MATLAB model.
net = MLPRegressor(
    hidden_layer_sizes=(12,),
    activation='tanh',
    solver='lbfgs',      # good default optimizer for small datasets like this one
    max_iter=2000,
    random_state=42,
)

net.fit(X_train, y_train)
print('Training complete.')


## Section 3: Check how well the network learned
This is the objective function's accuracy check -- if this is poor, the "optimum" found later is only optimal for a bad model of reality.

In [ ]:
predictions_test = net.predict(X_test)
r2 = r2_score(y_test, predictions_test)
print(f'Test R^2: {r2:.4f}')

plt.figure(figsize=(5, 5))
plt.scatter(y_test, predictions_test, alpha=0.7)
lims = [0, 1]
plt.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect fit (y = x)')
plt.xlabel('Actual normalized stress')
plt.ylabel('Predicted normalized stress')
plt.title(f'Predicted vs Actual (R^2 = {r2:.3f})')
plt.legend()
plt.show()


## Section 4: Define the objective function for optimization
**Important:** the network was trained on `[1/T(Kelvin), ln(strain rate), strain]`, not on raw `[temperature in Celsius, strain rate, strain]`. Optimizing directly over raw physical units would feed the network inputs it was never trained on, producing meaningless predictions. This function converts physical units to the network's expected features before calling it.

In [ ]:
scale_factor = 348.6897   # 1.0 (normalized) = 348.6897 MPa (this is sigma_max)

def predict_stress(x):
    """x = [temperature_C, strain_rate_per_s, strain] -> predicted stress in MPa"""
    T_celsius, strain_rate, strain = x
    T_kelvin = T_celsius + 273.15

    features = np.array([[1.0 / T_kelvin, np.log(strain_rate), strain]])
    normalized_stress = net.predict(features)[0]

    return normalized_stress * scale_factor   # convert back to MPa


## Section 5: Search for the forming parameters that minimize stress
`minimize(..., method='SLSQP')` finds a *local* minimum from a single starting point -- the Python equivalent of MATLAB's SQP algorithm. So we run it many times from random starting points ("multi-start") and keep the best result, to guard against getting stuck in a poor local minimum.

In [ ]:
lower_bounds = np.array([900, 0.1, 0.1])    # [deg C, per second, true strain]
upper_bounds = np.array([1050, 15.0, 0.7])
bounds = list(zip(lower_bounds, upper_bounds))

num_starts = 40
best_stress = np.inf
best_params = None

rng = np.random.default_rng(42)

for i in range(num_starts):
    x0 = rng.uniform(lower_bounds, upper_bounds)
    result = minimize(predict_stress, x0, method='SLSQP', bounds=bounds)

    if result.success and result.fun < best_stress:
        best_stress = result.fun
        best_params = result.x

print(f'Finished {num_starts} optimization runs.')


## Section 6: Report the result

In [ ]:
print('================ Optimization Result ================')
print(f'Best of {num_starts} random starting points:')
print(f'  Temperature : {best_params[0]:.2f} C')
print(f'  Strain rate : {best_params[1]:.4f} /s')
print(f'  Strain      : {best_params[2]:.4f}')
print(f'Minimum predicted stress: {best_stress:.4f} MPa')
print('=======================================================')
